# Exact Optimization Model (MIP / Exact)

## Purpose
Find the **exact optimal tote-entry order** for the current randomly generated data in `inputs/`.

## Decision Variables
- `x[i,j] in {0,1}`: 1 if tote `i` is immediately followed by tote `j` in the sequence.
- `s[i] in {0,1}`: 1 if tote `i` is the first tote.
- `e[i] in {0,1}`: 1 if tote `i` is the last tote.
- `u[i]` integer order index for subtour elimination (MTZ style).

## Objective (time-only exact benchmark)
Minimize total processing time:
- per-unit placement time
- internal bin-switch cost within each tote block
- inter-tote switch costs between consecutive totes

## Constraints
- Exactly one incoming arc per tote except the start tote.
- Exactly one outgoing arc per tote except the end tote.
- Exactly one start tote and one end tote.
- MTZ-style constraints to prevent subtours.
- Binary/integer domain constraints.

In [4]:
import csv
from pathlib import Path
import gurobipy as gp
from gurobipy import GRB

# Choose which generated input run(s) to use.
# - RUN_ID = None  -> canonical inputs/
# - RUN_ID = int   -> one run folder inputs/runs/run_XXXX
# - RUN_ID = "all" -> all run folders under inputs/runs/
RUN_ID = "all"


def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]


def _get_input_bases(run_id):
    if run_id == "all":
        runs_root = _resolve_existing([Path("inputs/runs"), Path("../inputs/runs")])
        if not runs_root.exists():
            return []
        return sorted([p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")])
    if run_id is None:
        return [_resolve_existing([Path("inputs"), Path("../inputs")])]
    run_name = f"run_{int(run_id):04d}"
    return [_resolve_existing([Path("inputs/runs") / run_name, Path("../inputs/runs") / run_name])]


output_base = _resolve_existing([Path("outputs"), Path("../outputs")])
output_base.mkdir(parents=True, exist_ok=True)
OUT = output_base

NUM_CONVEYORS = 4
PLACE_TIME = 1.75
TOTE_SWITCH_TIME = 4.0
BIN_SWITCH_TIME = 0.75

# Composite objective = total_time - OPTIONALITY_LAMBDA * optionality_score
OPTIONALITY_LAMBDA = 0.35
OPT_EDGE_WEIGHT = 1.0
OPT_BRANCH_WEIGHT = 0.6
OPT_RARE_WEIGHT = 0.4


def _coerce_int(v):
    s = v.strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except ValueError:
        return None


def _read_rows(p):
    with p.open("r", newline="") as f:
        return list(csv.reader(f))


def build_tasks():
    item_rows = _read_rows(INPUT_ITEMTYPES)
    qty_rows = _read_rows(INPUT_QUANTITIES)
    tote_rows = _read_rows(INPUT_TOTES)
    n_orders = max(len(item_rows), len(qty_rows), len(tote_rows))

    tasks = []
    for i in range(n_orders):
        ir = item_rows[i] if i < len(item_rows) else []
        qr = qty_rows[i] if i < len(qty_rows) else []
        tr = tote_rows[i] if i < len(tote_rows) else []
        width = max(len(ir), len(qr), len(tr))
        for j in range(width):
            item_type = _coerce_int(ir[j]) if j < len(ir) else None
            qty = _coerce_int(qr[j]) if j < len(qr) else None
            tote = _coerce_int(tr[j]) if j < len(tr) else None
            if item_type is None or qty is None or tote is None or qty <= 0:
                continue
            tasks.append({"order_id": i + 1, "item_type": item_type, "tote": tote, "qty": qty})
    return tasks


def build_tote_blocks(tasks):
    tote_to_bins = {}
    tote_to_items = {}
    for t in tasks:
        tote_to_bins.setdefault(t["tote"], []).extend([t["order_id"]] * t["qty"])
        tote_to_items.setdefault(t["tote"], []).extend([t["item_type"]] * t["qty"])

    blocks = {}
    for tote, bins in tote_to_bins.items():
        seq = sorted(bins)
        internal_bin_switches = sum(1 for k in range(1, len(seq)) if seq[k] != seq[k - 1])
        blocks[tote] = {
            "first_bin": seq[0],
            "last_bin": seq[-1],
            "units": len(seq),
            "internal_bin_switches": internal_bin_switches,
            "items": tote_to_items[tote],
        }
    return blocks


def edge_cost(i, j, blocks):
    c = TOTE_SWITCH_TIME
    if blocks[i]["last_bin"] != blocks[j]["first_bin"]:
        c += BIN_SWITCH_TIME
    return c


def fixed_block_cost(i, blocks):
    b = blocks[i]
    return b["units"] * PLACE_TIME + b["internal_bin_switches"] * BIN_SWITCH_TIME


def build_optionality_terms(blocks):
    totes = sorted(blocks.keys())
    first_bins = {t: blocks[t]["first_bin"] for t in totes}
    last_bins = {t: blocks[t]["last_bin"] for t in totes}

    compat = set()
    branch = {t: 0 for t in totes}

    for i in totes:
        for j in totes:
            if i == j:
                continue
            if last_bins[i] == first_bins[j]:
                compat.add((i, j))
                branch[i] += 1

    bin_freq = {}
    for t in totes:
        b = first_bins[t]
        bin_freq[b] = bin_freq.get(b, 0) + 1

    node_coeff = {}
    for t in totes:
        rarity = 1.0 / bin_freq[first_bins[t]]
        node_coeff[t] = OPT_BRANCH_WEIGHT * branch[t] - OPT_RARE_WEIGHT * rarity

    return {"compat": compat, "node_coeff": node_coeff}


def sequence_metrics(seq, blocks, terms):
    n = len(seq)
    total_time = 0.0
    optionality = 0.0

    prev = None
    for pos, tote in enumerate(seq, start=1):
        total_time += fixed_block_cost(tote, blocks)
        if prev is not None:
            total_time += edge_cost(prev, tote, blocks)
            if (prev, tote) in terms["compat"]:
                optionality += OPT_EDGE_WEIGHT
        optionality += terms["node_coeff"][tote] * (n + 1 - pos)
        prev = tote

    objective = total_time - OPTIONALITY_LAMBDA * optionality
    return total_time, optionality, objective


def solve_exact_mip(blocks, terms):
    totes = sorted(blocks.keys())
    n = len(totes)

    if n == 0:
        return [], 0.0, "empty"

    arcs = [(i, j) for i in totes for j in totes if i != j]

    model = gp.Model("tote_sequence_exact")
    model.Params.OutputFlag = 0

    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    s = model.addVars(totes, vtype=GRB.BINARY, name="s")
    e = model.addVars(totes, vtype=GRB.BINARY, name="e")
    u = model.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=n, name="u")

    n_float = float(n)

    fixed_term = gp.quicksum(fixed_block_cost(i, blocks) for i in totes)
    edge_term = gp.quicksum(edge_cost(i, j, blocks) * x[i, j] for (i, j) in arcs)

    node_optional = gp.quicksum(
        terms["node_coeff"][i] * (n_float + 1.0 - u[i]) for i in totes
    )

    compat_optional = gp.quicksum(
        OPT_EDGE_WEIGHT * x[i, j] for (i, j) in terms["compat"]
    )

    model.setObjective(
        fixed_term + edge_term - OPTIONALITY_LAMBDA * (node_optional + compat_optional),
        GRB.MINIMIZE,
    )

    for i in totes:
        model.addConstr(gp.quicksum(x[j, i] for j in totes if j != i) + s[i] == 1)

    for i in totes:
        model.addConstr(gp.quicksum(x[i, j] for j in totes if j != i) + e[i] == 1)

    model.addConstr(gp.quicksum(s[i] for i in totes) == 1)
    model.addConstr(gp.quicksum(e[i] for i in totes) == 1)

    for i, j in arcs:
        model.addConstr(u[i] - u[j] + n * x[i, j] <= n - 1)

    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError("Gurobi failed to find an optimal solution.")

    start = [i for i in totes if s[i].X > 0.5][0]

    seq = [start]
    cur = start
    visited = {start}

    while True:
        nxt = [j for j in totes if j != cur and x[cur, j].X > 0.5]
        if not nxt:
            break
        cur = nxt[0]
        if cur in visited:
            break
        seq.append(cur)
        visited.add(cur)

    return seq, float(model.ObjVal), "gurobi"


def build_sorter_input(seq, blocks):
    cols = {0: "circle", 1: "pentagon", 2: "trapezoid", 3: "triangle",
            4: "star", 5: "moon", 6: "heart", 7: "cross"}

    rows = {}
    for tote in seq:
        conv = ((blocks[tote]["first_bin"] - 1) % NUM_CONVEYORS) + 1
        rows.setdefault(conv, {name: 0 for name in cols.values()})
        for shape in blocks[tote]["items"]:
            if shape in cols:
                rows[conv][cols[shape]] += 1

    out = []
    for conv in sorted(rows.keys()):
        r = {"conv_num": conv}
        r.update(rows[conv])
        out.append(r)
    return out


def build_item_offload(seq, blocks):
    rows = []
    pos = 0
    for tote in seq:
        for item_type in blocks[tote]["items"]:
            pos += 1
            rows.append({"sequence_pos": pos, "item_type": item_type})
    return rows


def write_csv(path, fieldnames, rows):
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)


all_bases = _get_input_bases(RUN_ID)
if not all_bases:
    raise RuntimeError("No input runs found. Ensure inputs/runs exists or set RUN_ID appropriately.")

aggregate = []

for base in all_bases:
    INPUT_ITEMTYPES = base / "order_itemtypes.csv"
    INPUT_QUANTITIES = base / "order_quantities.csv"
    INPUT_TOTES = base / "orders_totes.csv"

    tasks = build_tasks()
    blocks = build_tote_blocks(tasks)
    terms = build_optionality_terms(blocks)

    seq, objective_score, solver_used = solve_exact_mip(blocks, terms)

    total_time, optionality_score, objective_check = sequence_metrics(seq, blocks, terms)

    run_out = OUT / "exact_mip_runs" / base.name
    run_out.mkdir(parents=True, exist_ok=True)

    write_csv(
        run_out / "exact_mip_tote_sequence.csv",
        ["sequence_pos", "tote"],
        [{"sequence_pos": i + 1, "tote": t} for i, t in enumerate(seq)],
    )

    sorter_rows = build_sorter_input(seq, blocks)
    write_csv(
        run_out / "optimized_input_from_exact_mip_model.csv",
        ["conv_num", "circle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"],
        sorter_rows,
    )

    write_csv(
        run_out / "exact_mip_tote_item_plan.csv",
        ["sequence_pos", "item_type"],
        build_item_offload(seq, blocks),
    )

    summary_row = {
        "run_name": base.name,
        "solver_used": solver_used,
        "total_time": total_time,
        "optionality_score": optionality_score,
        "objective_score": objective_check,
        "solver_objective": objective_score,
        "n_totes": len(seq),
        "n_units": sum(b["units"] for b in blocks.values()),
    }

    write_csv(run_out / "exact_mip_summary.csv", list(summary_row.keys()), [summary_row])
    aggregate.append(summary_row)

if RUN_ID == "all":
    write_csv(OUT / "exact_mip_all_runs_summary.csv", list(aggregate[0].keys()), aggregate)
    print(f"Processed {len(aggregate)} runs.")
    print("Wrote aggregate: outputs/exact_mip_all_runs_summary.csv")
else:
    print(f"Solved with: {aggregate[0]['solver_used']}")
    print(f"Exact optimal objective: {aggregate[0]['objective_score']:.3f}")
    print(f"Total time component: {aggregate[0]['total_time']:.3f}")
    print("Wrote run outputs under outputs/")

Processed 500 runs.
Wrote aggregate: outputs/exact_mip_all_runs_summary.csv
